# 第5章 收益率与风险度量 —— 配套代码

对应正文 `book/part2/05-returns-risk.md`。依赖内置数据，离线可跑。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fds import (load_sample_prices, daily_returns, set_chinese_font,
                 annualized_return, annualized_volatility, sharpe_ratio, max_drawdown)

set_chinese_font()
prices = load_sample_prices()
rets = daily_returns(prices)
rets.tail()

## 风险收益总览表

In [ ]:
table = pd.DataFrame({
    "年化收益": annualized_return(rets),
    "年化波动": annualized_volatility(rets),
    "夏普(rf=2%)": sharpe_ratio(rets, risk_free=0.02),
    "最大回撤": {c: max_drawdown(rets[c]) for c in rets.columns},
})
table.sort_values("夏普(rf=2%)", ascending=False).round(3)

## 累计净值与回撤曲线

In [ ]:
nav = (1 + rets["TECH"]).cumprod()
drawdown = nav / nav.cummax() - 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
nav.plot(ax=ax1, title="TECH 累计净值"); ax1.set_ylabel("净值")
drawdown.plot(ax=ax2, color="crimson", title="TECH 回撤")
ax2.fill_between(drawdown.index, drawdown.values, 0, color="crimson", alpha=0.3)
ax2.set_ylabel("回撤"); ax2.set_xlabel("日期")
plt.tight_layout(); plt.show()
print("最大回撤：", round(max_drawdown(rets["TECH"]), 3))

## VaR 与 ES：历史模拟法 vs 正态参数法

In [ ]:
from scipy.stats import norm

x = rets["TECH"]
def var_es(series, alpha=0.95):
    q = np.quantile(series, 1 - alpha)
    hist_var = -q
    hist_es = -series[series <= q].mean()
    z = norm.ppf(1 - alpha)
    norm_var = -(series.mean() + z * series.std(ddof=1))
    return hist_var, hist_es, norm_var

for a in (0.95, 0.99):
    hv, he, nv = var_es(x, a)
    print(f"{int(a*100)}% 单日：历史VaR={hv:.4f}  历史ES={he:.4f}  正态VaR={nv:.4f}")

In [ ]:
# 直方图标注 95% VaR
var95 = -np.quantile(x, 0.05)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(x, bins=50, alpha=0.7)
ax.axvline(-var95, color="crimson", lw=2, label=f"95% VaR = {var95:.3f}")
ax.set_title("TECH 日收益分布与 95% VaR"); ax.set_xlabel("日收益率"); ax.legend()
plt.tight_layout(); plt.show()